In [26]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [27]:
from ultralytics import YOLO
from ultralytics.utils import YAML

In [28]:
model_version = "yolo26s.pt"
yaml_path = '/kaggle/input/datasets/sh1nd3l/yolo-coin-dataset/coin_yolo_split/data.yaml'
model = YOLO(model_version)

# Param Grid

In [30]:
search_space = {
    "lr0": (1e-4, 5e-3),
    "lrf": (0.05, 0.5),
    "weight_decay": (1e-5, 1e-3),

    "mosaic": (0.4, 0.9),
    "mixup": (0.0, 0.10),
    "close_mosaic": (10.0, 20.0),

    "degrees": (5.0, 20.0),
    "scale": (0.3, 0.7),
    "translate": (0.03, 0.15),
    "shear": (0.0, 4.0),
    "perspective": (0.0, 0.001),

    "hsv_h": (0.005, 0.03),
    "hsv_s": (0.3, 0.8),
    "hsv_v": (0.2, 0.5),
    "fliplr": (0.2, 0.6),
    "flipud": (0.0, 0.15),
}


# Model tune ( HyperParameters searching )

In [31]:
model.tune(
    data=yaml_path,
    epochs=60, # Epochs Per Iterations
    iterations=15, # Total Number of Iterations

    imgsz=640,
    batch=32,
    device=[0, 1],
    patience=15,
    cache="ram",
    workers=4,
    amp=True,
    optimizer="AdamW",

    project="coin_detector",
    name="yolo26n_coin_tune",

    space=search_space, # Param Grid
    plots=False,
    save=False,
    val=False,
)

Tuner: Initialized Tuner instance with 'tune_dir=/kaggle/working/runs/detect/coin_detector/yolo26n_coin_tune'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/15 with hyperparameters: {'lr0': 0.005, 'lrf': 0.05, 'weight_decay': 0.0005, 'mosaic': 0.9, 'mixup': 0.0, 'close_mosaic': 10, 'degrees': 5.0, 'scale': 0.5, 'translate': 0.1, 'shear': 0.0, 'perspective': 0.0, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'fliplr': 0.5, 'flipud': 0.0}
Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/sh1nd3l/yolo-coin-da

# Final Train

In [34]:
model = YOLO(model_version)
path_to_params = '/kaggle/working/runs/detect/coin_detector/yolo26n_coin_tune/best_hyperparameters.yaml'
best_hyp = YAML.load(path_to_params)

results = model.train(
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,
    device=[0, 1],
    patience=15,
    cache='disk',
    workers=4,
    amp=True,
    project="coin_detector",
    name="yolo26n_coin_multiobj_640_tuned",
    verbose=True,
    **best_hyp,
)

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=12, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/sh1nd3l/yolo-coin-dataset/coin_yolo_split/data.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.40434, flipud=0.0032, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01233, hsv_s=0.70924, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00274, lrf=0.05107, mask_ratio=4, max_det=300, mixup=0.00409, mode=train, model=yolo26s.pt, momentum=0.937